# RPY Shear Simulation Tutorial

This notebook demonstrates how to run Brownian Dynamics simulations of colloidal suspensions using the **Rotne-Prager-Yamakawa (RPY)** hydrodynamic tensor with **shear flow** (Lees-Edwards boundary conditions).

## Overview
- Setting up a suspension of spherical particles
- Applying simple shear flow with periodic boundaries
- Using the spectral Ewald RPY method for hydrodynamic interactions
- Computing stress tensors and analyzing rheology
- Visualizing trajectories with periodic images

## Key physics
The RPY tensor describes hydrodynamic interactions between spheres in a viscous fluid:
$$\mathbf{v}_i = \sum_j \mathbf{M}_{ij} \cdot \mathbf{F}_j + \sqrt{2 k_B T} \sum_j \mathbf{B}_{ij} \cdot \mathbf{W}_j$$
where $\mathbf{M}$ is the mobility tensor, $\mathbf{B}\mathbf{B}^T = \mathbf{M}$ (fluctuation-dissipation), and $\mathbf{W}$ is white noise.

## Precision: f32 vs f64

Set `jax_enable_x64 = True` (below) for f64, or `False` for f32.

In [ ]:
# Choose precision for JAX computations.
from jax import config
config.update("jax_enable_x64", False)

import math
import os

import jax
import jax.numpy as jnp
import numpy as np
from jax import random
import matplotlib.pyplot as plt

from jax_md import energy, minimize, partition, space, simulate, smap, util, rheo
from jax_md.hydro import rpy

print(f"JAX devices: {jax.devices()}")

## 1. System Setup

We define the basic simulation parameters. Key quantities:

- **N**: Number of particles
- **φ (phi)**: Volume fraction
- **a**: Particle radius (sets the length scale)
- **η (eta)**: Solvent viscosity
- **kT**: Thermal energy (sets the energy scale)
- **Pe (Peclet number)**: Ratio of advective to diffusive transport = $\dot{\gamma} a^2 / 2D_0$, where $D_0 = k_B T / (6\pi\eta a)$

In [ ]:
# ============================================================
# SIMULATION PARAMETERS - Modify these to your needs
# ============================================================

# System parameters
N = 128                    # Number of particles
phi = 0.45                 # Volume fraction
dim = 3                   # Dimension (must be 3 for RPY)

# Physical parameters
a = 1.0                   # Particle radius
diameter = 2.0 * a        # Particle diameter
eta = 1.0 / (6 * math.pi) # Solvent viscosity, set so that D0 = kT/(6*pi*eta*a) = 1.0
kT = 1.0                  # Thermal energy

# Shear parameters
peclet = 1.0              # Peclet number (0 = equilibrium, >0 = sheared)
# Alternatively, set shear_rate directly:
# shear_rate = 0.1        # Direct shear rate

# Time integration
dt = 2e-5                  # Timestep (decrease with increasing Pe or density)
n_steps = 30000            # Total simulation steps

# RPY Ewald parameters (xi should be between 0.1 and 1.0 for typical systems)
xi = 0.7                  # Ewald splitting parameter
tol = 1e-4                # Target tolerance for RPY accuracy

# Potential parameters (soft inverse-power repulsion to prevent overlaps)
beta_eps = 100.0          # Prefactor for beta*u(r) = beta_eps * (sigma/r)^exponent
sigma = 2.0 * a * 0.95    # Effective particle diameter (Barker-Henderson for these parameters)
exponent = 96.0           # Steepness (higher = harder)
r_cut = 1.5 * sigma       # Cutoff radius (for numerical efficiency)

# Output control
stress_every = 10         # Record stress every N steps
traj_every = 100          # Record trajectory every N steps (0 to disable)

# Thermalization
thermalize_steps = 10000   # Equilibration steps before shear (no shear applied)

# Random seed
seed = 42

### Compute derived quantities

In [ ]:
def box_size_from_phi(n_particles, radius, phi, dim=3):
    """Compute cubic box size for given volume fraction."""
    if dim == 3:
        volume = n_particles * (4.0 / 3.0) * np.pi * radius ** 3 / phi
        return volume ** (1.0 / 3.0)
    elif dim == 2:
        area = n_particles * np.pi * radius ** 2 / phi
        return np.sqrt(area)
    raise ValueError(f"Unsupported dim={dim}")

# Box size
L = box_size_from_phi(N, a, phi, dim)
base_box = jnp.eye(dim) * L

# Compute shear rate from Peclet number
D0 = kT / (6.0 * math.pi * eta * a)  # Stokes-Einstein diffusivity
shear_rate = 2.0 * peclet * D0 / (a ** 2) # Brady's definition of Pe uses 2D0 in denominator

print(f"Box size L = {L:.4f}")
print(f"Diffusion coefficient D0 = {D0:.4e}")
print(f"Shear rate = {shear_rate:.4e}")
print(f"Peclet number = {peclet:.2f}")
print(f"Strain per step = {shear_rate * dt:.4e}")

## 2. Define Space and Shear Schedule

JAX-MD uses **space functions** to handle periodic boundaries and coordinate transformations.

### Lees-Edwards Boundary Conditions
Under simple shear, the simulation box deforms over time:
- The top of the box moves in the +x direction relative to the bottom
- Strain accumulates as $\gamma(t) = \dot{\gamma} \cdot t$

When `remap=True`, the strain is wrapped to $[-0.5, 0.5]$ to keep the box well-conditioned. This causes periodic "remaps" where particle fractional coordinates jump.

### Fractional vs Real Coordinates
We use **fractional coordinates** where positions are in $[0,1]^3$. The box matrix transforms these to real space:
$$\mathbf{r}_{real} = \mathbf{H} \cdot \mathbf{r}_{frac}$$

In [ ]:
# Define shear schedule: gamma(t) = shear_rate * t
shear_fn = lambda t: shear_rate * t

# Create shearing space functions
# fractional_coordinates=True means positions are in [0,1]^3
# remap=True handles Lees-Edwards boundary conditions automatically
displacement, shift, box_of = space.shearing(
    base_box, 
    shear_schedule=shear_fn, 
    fractional_coordinates=True, 
    remap=True
)

# Also create static space functions for relaxation/equilibrium
displacement_0, shift_0 = space.periodic_general(base_box, fractional_coordinates=True)

print("Space functions created.")

## 3. Define the Pair Potential

We need a repulsive potential to prevent particle overlaps. The **inverse-power potential** is a common choice:

$$u(r) = \epsilon \left(\frac{\sigma}{r}\right)^{n}$$

where:
- $\sigma$ is the contact diameter
- $\epsilon$ controls the energy scale
- $n$ (exponent) controls steepness — higher = harder spheres

In [ ]:
def inverse_power(dr, epsilon, sigma, exponent, r_cut=None, **unused):
    """
    Steep inverse-power pair potential: u(r) = epsilon * (sigma/r)^exponent
    
    This implementation is safe for both f32 and f64, and handles dr=0
    (self-interactions) gracefully. The key is to ensure no inf/nan values
    are produced even in masked-out regions, because JAX computes gradients
    through all branches.
    """
    dr = jnp.asarray(dr)
    
    # Clamp dr to sigma to prevent overflow (log(sigma/sigma) = 0, exp(0) = 1)
    dr_safe = jnp.maximum(dr, sigma)
    
    # Compute potential - safe because dr_safe >= sigma means ratio <= 1
    ratio = sigma / dr_safe
    log_term = exponent * jnp.log(ratio)
    max_log = 85.0  # Safe for f32 (exp(88) overflows)
    val = epsilon * jnp.exp(jnp.minimum(log_term, max_log))
    
    # Apply mask for self-interactions and cutoff
    mask = (dr > 0.0)
    if r_cut is not None:
        mask = mask & (dr < r_cut)
    
    return jnp.where(mask, val, 0.0)

# Convert to energy units
epsilon = beta_eps * kT

# Create pair energy functions
# For shearing simulation
energy_fn = smap.pair(
    inverse_power,
    space.canonicalize_displacement_or_metric(displacement),
    ignore_unused_parameters=True,
    sigma=sigma,
    epsilon=epsilon,
    exponent=exponent,
    r_cut=r_cut,
)

# For equilibrium/relaxation
energy_fn_0 = smap.pair(
    inverse_power,
    space.canonicalize_displacement_or_metric(displacement_0),
    ignore_unused_parameters=True,
    sigma=sigma,
    epsilon=epsilon,
    exponent=exponent,
    r_cut=r_cut,
)

print(f"Potential: epsilon={epsilon:.2f}, sigma={sigma:.2f}, exponent={exponent:.0f}")
print(f"Precision: {jnp.array(1.0).dtype}")

In [ ]:
# Plot the inverse-power potential
r_plot = np.linspace(0.8 * sigma, 2.0 * sigma, 200)
u_plot = np.array([float(inverse_power(r, epsilon, sigma*0.97, exponent, r_cut)) for r in r_plot])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
ax = axes[0]
ax.plot(r_plot, u_plot / kT, 'b-', lw=2)
ax.axvline(2.0, color='r', ls='--', label=r'$r = 2a$')
ax.axvline(r_cut, color='g', ls=':', label=f'$r_{{cut}} = {r_cut:.2f} a$')
ax.set_xlabel(r'$r$', fontsize=12)
ax.set_ylabel(r'$\beta u(r)$', fontsize=12)
ax.set_title('Inverse-Power Potential (linear scale)')
ax.set_xlim(0.8*sigma, 2.0*sigma)
ax.set_ylim(0, 50)
ax.legend()
ax.grid(True, alpha=0.3)

# Log scale
ax = axes[1]
ax.semilogy(r_plot, u_plot / kT, 'b-', lw=2)
ax.axvline(2.0, color='r', ls='--', label=r'$r = 2a$')
ax.axvline(r_cut, color='g', ls=':', label=f'$r_{{cut}} = {r_cut:.2f} a$')
ax.set_xlabel(r'$r$', fontsize=12)
ax.set_ylabel(r'$\beta u(r)$', fontsize=12)
ax.set_title('Inverse-Power Potential (log scale)')
ax.set_xlim(0.8*sigma, 2.0*sigma)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At r=sigma: beta*u = {float(inverse_power(sigma, epsilon, sigma, exponent, r_cut))/kT:.2e}")
print(f"At r=1.05*sigma: beta*u = {float(inverse_power(1.05*sigma, epsilon, sigma, exponent, r_cut))/kT:.2e}")
print(f"At r=r_cut: beta*u = {float(inverse_power(r_cut, epsilon, sigma, exponent, r_cut))/kT:.2e}")

## 4. Initialize and Relax Positions

We start with random positions in the box. **CRITICAL**: Random positions will have overlaps that cause numerical instability in the RPY calculation.

### FIRE minimization
We use the **FIRE** (Fast Inertial Relaxation Engine) algorithm to push particles apart. It's a damped dynamics method that efficiently finds local energy minima.

After relaxation:
- All particle pairs should have $r > \sigma$
- Total energy should be small (ideally ~0)

In [ ]:
key = random.PRNGKey(seed)
key, init_key = random.split(key)

# Random initial positions in fractional coordinates [0,1]^3
positions = random.uniform(init_key, (N, dim), dtype=jnp.float64)

print(f"Initial positions shape: {positions.shape}")
print(f"Position range: [{float(positions.min()):.3f}, {float(positions.max()):.3f}]")

In [ ]:
# CRITICAL: Relax positions to remove overlaps
# Without this step, the simulation will produce NaNs!

fire_steps = 250  # Adjust this if overlaps remain after relaxation

def relax_positions(R_init, displacement_fn, shift_fn, diameter, box, steps=500):
    """Use FIRE minimization to remove particle overlaps."""
    if steps <= 0:
        return R_init
    
    # Create soft-sphere energy with neighbor list for efficiency
    neighbor_fn, energy_relax = energy.soft_sphere_neighbor_list(
        displacement_fn,
        box,
        sigma=diameter,  # Slightly larger than diameter to properly relax overlaps
        epsilon=1.0,
        alpha=2.0,
        dr_threshold=0.2,
        fractional_coordinates=True,
        format=partition.NeighborListFormat.Sparse,
        capacity_multiplier=2.0,
    )
    neighbor = neighbor_fn.allocate(R_init, box=box)
    
    init_min, apply_min = minimize.fire_descent(energy_relax, shift_fn)
    state = init_min(R_init, neighbor=neighbor)
    
    for i in range(steps):
        neighbor = neighbor.update(state.position, box=box)
        if neighbor.did_buffer_overflow:
            neighbor = neighbor_fn.allocate(state.position, box=box)
        state = apply_min(state, neighbor=neighbor)
    
    return state.position

def count_overlaps(R, displacement_fn, diameter):
    """Count particle pairs closer than diameter."""
    # Compute pairwise displacements
    dR = space.map_product(displacement_fn)(R, R)  # (N, N, dim)
    dists = space.distance(dR)  # (N, N)
    # Upper triangle excluding diagonal (to count each pair once)
    mask = jnp.triu(jnp.ones((len(R), len(R)), dtype=bool), k=1)
    overlaps = (dists < diameter) & mask
    n_overlaps = int(jnp.sum(overlaps))
    # Min distance excluding self
    dists_no_self = jnp.where(jnp.eye(len(R), dtype=bool), jnp.inf, dists)
    min_dist = float(jnp.min(dists_no_self))
    return n_overlaps, min_dist

# Check overlaps BEFORE relaxation
n_overlaps_before, min_dist_before = count_overlaps(positions, displacement_0, diameter)
E_before = energy_fn_0(positions)
print(f"BEFORE relaxation:")
print(f"  Overlapping pairs (r < 2a): {n_overlaps_before}")
print(f"  Minimum distance: {min_dist_before:.4f}")
print(f"  Energy: {E_before:.4e}")

# Relax
print("\nRelaxing positions...")
positions = relax_positions(positions, displacement_0, shift_0, diameter, base_box, steps=fire_steps)

# Check overlaps AFTER relaxation
n_overlaps_after, min_dist_after = count_overlaps(positions, displacement_0, diameter)
E_after = energy_fn_0(positions)
print(f"\nAFTER relaxation:")
print(f"  Overlapping pairs (r < 2a): {n_overlaps_after}")
print(f"  Minimum distance: {min_dist_after:.4f}")
print(f"  Energy: {E_after:.4e}")

if n_overlaps_after > 0 or E_after > 1e6:
    print("\nWARNING: Overlaps remain after relaxation!")
    print("Try: (1) more relax steps, (2) lower phi, or (3) check sigma value")
else:
    print("\nRelaxation successful - no overlaps, ready for simulation.")

## 5. Configure RPY Parameters

The RPY mobility is computed using **Spectral Ewald summation**, which splits the calculation into:
- **Real-space sum**: Short-range, computed with neighbor lists
- **Wave-space sum**: Long-range, computed with FFT on a grid

### Key parameters:
- **ξ (xi)**: Ewald splitting parameter. Larger ξ = more work in wave space
- **rcut**: Real-space cutoff. Pairs with $r > r_{cut}$ are handled in wave space ONLY
- **P**: Number of support points for spreading/interpolation
- **M (Mgrid)**: Grid size for FFT (should be power of 2 for efficiency)

The `rpy.estimate_rpy_params()` function automatically selects these based on:
- Target accuracy (tolerance)
- Box size and particle count
- Volume fraction

See Fiore et al. (2017) for details on parameter selection.

In [ ]:
# Estimate RPY parameters
rpy_params = rpy.estimate_rpy_params(
    tol=tol,
    A=base_box,
    a=a,
    N=N,
    phi=phi,
    xi_override=xi,
    notes=True,  # Include diagnostic info
)

print("\nRPY Parameters:")
print(f"  xi (Ewald split)   = {rpy_params.xi:.4f}")
print(f"  rcut (real-space)  = {rpy_params.rcut:.4f}")
print(f"  P (quadrature)     = {rpy_params.P}")
print(f"  M (FFT grid)       = {rpy_params.M}")
print(f"  theta (Gaussian)   = {rpy_params.theta:.4f}")

if rpy_params.diagnostics is not None:
    notes = rpy_params.diagnostics
    print(f"\nError estimates:")
    print(f"  eps_r (real)  = {notes.eps_r_est:.2e}")
    print(f"  eps_w (wave)  = {notes.eps_w_est:.2e}")
    print(f"  eps_q (quad)  = {notes.eps_q_est:.2e}")

## 6. Create the RPY Shear Integrator

The integrator advances particles in time according to:
$$\mathbf{r}(t + \Delta t) = \mathbf{r}(t) + \mathbf{M} \cdot \mathbf{F} \cdot \Delta t + \sqrt{2 k_B T \Delta t} \, \mathbf{B} \cdot \mathbf{W}$$

where the stochastic term satisfies the fluctuation-dissipation theorem.
This is done using the `simulate.rpy_with_shear()` function.

### Neighbor lists for real-space RPY
Neighbor-list controls are passed as explicit kwargs (`neighbor_format`, `dr_threshold`, `capacity_multiplier`):
- `Dense`: Fastest for small N, stores full neighbor matrix
- `Sparse`: Memory-efficient for large N, stores only neighbor pairs

In [ ]:
# Neighbor-list parameters for real-space RPY
# Dense format is fastest for small systems; Sparse uses less memory for large N.
neighbor_format = partition.NeighborListFormat.Sparse  # or Dense
rpy_dr_threshold = 0.2          # Rebuild threshold
rpy_capacity_multiplier = 1.5   # Extra capacity for neighbor list

# rpy_with_shear now expects a 3-component shear-vector schedule.
shear_vector_fn = lambda t: (shear_fn(t), 0.0, 0.0)

# Create the RPY integrator with shear
init_fn, apply_fn = simulate.rpy_with_shear(
    (displacement, shift, box_of),  # Space functions
    energy_fn,                      # Energy/force function
    dt=dt,
    kT=kT,
    a=a,
    xi=xi,
    eta=eta,
    shear_vector_schedule=shear_vector_fn,
    neighbor_format=neighbor_format,
    dr_threshold=rpy_dr_threshold,
    capacity_multiplier=rpy_capacity_multiplier,
    rcut=rpy_params.rcut,
    P=rpy_params.P,
    Mgrid=rpy_params.M,
    theta=rpy_params.theta,
)

# Create equilibrium RPY integrator (no shear) for thermalization
init_fn_eq, apply_fn_eq = simulate.rpy(
    (displacement_0, shift_0),      # Static space functions
    energy_fn_0,                    # Equilibrium energy function
    dt=dt,
    kT=kT,
    a=a,
    xi=xi,
    eta=eta,
    neighbor_format=neighbor_format,
    dr_threshold=rpy_dr_threshold,
    capacity_multiplier=rpy_capacity_multiplier,
    rcut=rpy_params.rcut,
    P=rpy_params.P,
    Mgrid=rpy_params.M,
    theta=rpy_params.theta,
)

print("RPY integrators created (shear + equilibrium).")
print(f"Real-space neighbor format: {neighbor_format}")


## 7. Create Stress Tensor Function

To analyze rheology, we need the **stress tensor**. For pairwise interactions, the Irving-Kirkwood formula gives:

$$\sigma_{\alpha\beta} = -\frac{1}{V} \sum_{i<j} r_{ij,\alpha} \, F_{ij,\beta}$$

The **shear stress** $\sigma_{xy}$ is particularly important:
- It measures resistance to shear flow
- The **relative viscosity** is $\eta_r = \sigma_{xy} / (\eta \dot{\gamma})$

The `rheo.make_pairwise_stress_fn()` creates a function that computes the full stress tensor from particle positions.

In [ ]:
# Create stress calculation function
stress_fn = rheo.make_pairwise_stress_fn(
    inverse_power,
    sigma=sigma,
    epsilon=epsilon,
    exponent=exponent,
    r_cut=r_cut,
)

print("Stress function created.")

## 8. Run the Simulation

The simulation has two phases:

### Phase 1: Thermalization (no shear)
Run equilibrium Brownian dynamics to let the system relax and sample configurations. This helps ensure the system is well-equilibrated before applying shear.

### Phase 2: Shear production run
Apply shear and collect data:
- Stress tensor (for viscosity)
- Particle trajectories (for visualization)
- Monitor for NaN values (indicates numerical instability)

In [ ]:
# ============================================================
# THERMALIZATION (equilibrium, no shear)
# ============================================================
key, therm_key = random.split(key)
state_eq = init_fn_eq(therm_key, positions)

apply_fn_eq_jit = jax.jit(apply_fn_eq)

if thermalize_steps > 0:
    print(f"Thermalizing for {thermalize_steps} steps (no shear)...")
    for step in range(thermalize_steps):
        if step % 200 == 0:
            print(f"  Thermalization step {step}/{thermalize_steps}")
        
        # Check for NaNs
        if jnp.any(jnp.isnan(state_eq.mobility_position)):
            print(f"ERROR: NaN detected during thermalization at step {step}!")
            break
        
        state_eq = apply_fn_eq_jit(state_eq)
    
    # Use thermalized positions for shear run
    positions_thermalized = state_eq.mobility_position
    print(f"Thermalization complete.")
    
    # Check overlaps after thermalization
    n_overlaps_therm, min_dist_therm = count_overlaps(positions_thermalized, displacement_0, sigma)
    print(f"After thermalization: {n_overlaps_therm} overlaps, min_dist={min_dist_therm:.4f}")
else:
    positions_thermalized = positions
    print("Skipping thermalization (thermalize_steps = 0).")

In [ ]:
# ============================================================
# SHEAR SIMULATION
# ============================================================
key, run_key = random.split(key)
state = init_fn(run_key, positions_thermalized)

# Storage for results
times = []
strains = []
stresses = []
trajectories = []

# JIT compile the apply function for speed
apply_fn_jit = jax.jit(apply_fn)

print(f"Running shear simulation for {n_steps} steps...")
print(f"Total strain: {shear_rate * n_steps * dt:.4f}")

for step in range(n_steps):
    t = state.time
    gamma = shear_fn(t)
    
    # Record stress
    if step % stress_every == 0:
        current_box = box_of(t=t)
        stress_tensor = stress_fn(
            state.mobility_position,
            box=current_box,
            fractional_coordinates=True,
        )
        times.append(float(t))
        strains.append(float(gamma))
        stresses.append(np.array(stress_tensor))
    
    # Record trajectory
    if traj_every > 0 and step % traj_every == 0:
        trajectories.append({
            'time': float(t),
            'strain': float(gamma),
            'positions_frac': np.array(state.mobility_position),
            'positions_real': np.array(state.position),
        })
    
    # Progress
    if step % 500 == 0:
        print(f"  Step {step}/{n_steps}, t={float(t):.4f}, gamma={float(gamma):.4f}")
    
    # Check for NaNs
    if jnp.any(jnp.isnan(state.mobility_position)):
        print(f"ERROR: NaN detected at step {step}! Stopping.")
        break
    
    # Advance one step
    state = apply_fn_jit(state)

print("Simulation complete!")

# Convert to arrays
times = np.array(times)
strains = np.array(strains)
stresses = np.array(stresses)

## 9. Analyze Results

### 9.1 Stress vs. Strain

For steady shear flow, the stress should fluctuate around a mean value. The ratio:
$$\eta_r = \frac{\langle \sigma_{xy} \rangle}{\eta \dot{\gamma}}$$
gives the **relative viscosity**, which is how much the particles increase the fluid's effective viscosity.

In [ ]:
# Extract shear stress (xy component)
sigma_xy = stresses[:, 0, 1]

# For suspensions under steady shear, the relative viscosity is:
# eta_r = sigma_xy / (eta * shear_rate)
if shear_rate > 0:
    eta_rel = sigma_xy / (eta * shear_rate)
    mean_eta_rel = np.mean(eta_rel[len(eta_rel)//2:])  # Average over second half
    print(f"Mean relative viscosity (second half): {mean_eta_rel:.4f}")

# Plot stress vs strain
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Shear stress
ax = axes[0]
ax.plot(strains, sigma_xy, 'b-', alpha=0.7)
ax.set_xlabel('Strain $\gamma$')
ax.set_ylabel('Shear stress $\sigma_{xy}$')
ax.set_title('Shear Stress vs Strain')
ax.axhline(0, color='k', linestyle='--', alpha=0.3)

# Relative viscosity
if shear_rate > 0:
    ax = axes[1]
    ax.plot(strains, eta_rel, 'r-', alpha=0.7)
    ax.axhline(mean_eta_rel, color='k', linestyle='--', label=f'Mean = {mean_eta_rel:.3f}')
    ax.set_xlabel('Strain $\gamma$')
    ax.set_ylabel('Relative viscosity $\eta_r$')
    ax.set_title('Relative Viscosity vs Strain')
    ax.legend()

plt.tight_layout()
plt.show()

### 9.2 Full Stress Tensor

In [ ]:
# Plot all stress components
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

labels = [['xx', 'xy', 'xz'], ['yx', 'yy', 'yz']]
for i in range(2):
    for j in range(3):
        ax = axes[i, j]
        component = stresses[:, i, j]
        ax.plot(strains, component, alpha=0.7)
        ax.set_xlabel('Strain $\gamma$')
        ax.set_ylabel(f'$\\sigma_{{{labels[i][j]}}}$')
        ax.set_title(f'Stress component {labels[i][j]}')
        ax.axhline(0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### 9.3 Trajectory Visualization

In [ ]:
if len(trajectories) > 0:
    # Plot initial and final configurations
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Initial
    ax = axes[0]
    pos_init = trajectories[0]['positions_frac']
    ax.scatter(pos_init[:, 0] * L, pos_init[:, 1] * L, s=50, alpha=0.7)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Initial (t={trajectories[0]["time"]:.3f}, $\\gamma$={trajectories[0]["strain"]:.3f})')
    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect('equal')
    
    # Final
    ax = axes[1]
    pos_final = trajectories[-1]['positions_frac']
    ax.scatter(pos_final[:, 0] * L, pos_final[:, 1] * L, s=50, alpha=0.7, c='red')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Final (t={trajectories[-1]["time"]:.3f}, $\\gamma$={trajectories[-1]["strain"]:.3f})')
    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
else:
    print("No trajectories saved. Set traj_every > 0 to enable.")

### 9.4 Trajectory Movie

When **REMAP!** flashes, the box and particles have undergone a Lees-Edwards coordinate transformation to keep the strain bounded.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Image, display

if len(trajectories) > 1:
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Image offsets in fractional coordinates
    image_offsets = [(-1, -1), (0, -1), (1, -1),
                     (-1,  0), (0,  0), (1,  0),
                     (-1,  1), (0,  1), (1,  1)]
    
    # Create scatter plots
    scatter_main = ax.scatter([], [], s=120, alpha=0.9, c='steelblue', 
                              edgecolors='k', lw=0.5, zorder=10, label='Primary cell')
    
    scatter_images = []
    for ox, oy in image_offsets:
        if ox == 0 and oy == 0:
            continue
        sc = ax.scatter([], [], s=80, alpha=0.25, c='lightblue', 
                       edgecolors='gray', lw=0.2, zorder=5)
        scatter_images.append((ox, oy, sc))
    
    # Box outlines - show TOTAL strain box in addition to reduced
    box_main, = ax.plot([], [], 'k-', lw=2.5, zorder=8, label='Primary box (reduced γ)')
    box_total, = ax.plot([], [], 'r--', lw=2, alpha=0.6, zorder=7, label='Total strain box')
    
    box_images = []
    for ox, oy in image_offsets:
        if ox == 0 and oy == 0:
            continue
        line, = ax.plot([], [], 'gray', lw=1, alpha=0.4, ls='--', zorder=3)
        box_images.append((ox, oy, line))
    
    # Remap indicator
    remap_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, fontsize=14,
                         color='red', fontweight='bold', va='top', ha='left', 
                         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    
    title = ax.set_title('', fontsize=12)
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_xlim(-1.3 * L, 2.3 * L)
    ax.set_ylim(-0.4 * L, 1.4 * L)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right', fontsize=9)
    
    # Track previous reduced strain for remap detection
    prev_gamma_reduced = [0.0]
    remap_counter = [0]  # Counts frames since last remap (for flash duration)
    
    def reduce_strain(gamma_total):
        """Reduce strain to [-0.5, 0.5) range (Lees-Edwards remap)."""
        return gamma_total - np.floor(gamma_total + 0.5)
    
    def fractional_to_real_sheared(pos_frac, gamma, offset_x=0, offset_y=0):
        """Convert fractional [0,1] coordinates to real sheared coordinates."""
        frac_with_offset = pos_frac.copy()
        frac_with_offset[:, 0] += offset_x
        frac_with_offset[:, 1] += offset_y
        
        x_real = frac_with_offset[:, 0] * L + frac_with_offset[:, 1] * gamma * L
        y_real = frac_with_offset[:, 1] * L
        
        return x_real, y_real
    
    def draw_box_corners(gamma, offset_x=0, offset_y=0):
        """Draw corners of a sheared box."""
        corners_frac = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]])
        corners_frac[:, 0] += offset_x
        corners_frac[:, 1] += offset_y
        
        x = corners_frac[:, 0] * L + corners_frac[:, 1] * gamma * L
        y = corners_frac[:, 1] * L
        return x, y
    
    def update(frame):
        traj = trajectories[frame]
        pos_frac = traj['positions_frac']
        t = traj['time']
        gamma_total = traj['strain']  # This is shear_rate * t (total strain)
        
        # Compute reduced strain (what the simulation actually uses with remap=True)
        gamma_reduced = reduce_strain(gamma_total)
        
        # Detect remap: when reduced strain jumps discontinuously
        gamma_diff = gamma_reduced - prev_gamma_reduced[0]
        remapped = abs(gamma_diff) > 0.4 and frame > 0
        prev_gamma_reduced[0] = gamma_reduced
        
        if remapped:
            remap_counter[0] = 0  # Reset flash counter
        else:
            remap_counter[0] += 1
        
        # Update primary cell particles (use reduced strain)
        x_main, y_main = fractional_to_real_sheared(pos_frac, gamma_reduced, 0, 0)
        scatter_main.set_offsets(np.column_stack([x_main, y_main]))
        
        # Update periodic image particles (use reduced strain)
        for ox, oy, sc in scatter_images:
            x_img, y_img = fractional_to_real_sheared(pos_frac, gamma_reduced, ox, oy)
            sc.set_offsets(np.column_stack([x_img, y_img]))
        
        # Update primary box with REDUCED strain (black, solid) - this will JUMP at remap!
        bx_reduced, by_reduced = draw_box_corners(gamma_reduced, 0, 0)
        box_main.set_data(bx_reduced, by_reduced)
        
        # Show TOTAL strain box (red, dashed) - this shears continuously, no jump
        bx_total, by_total = draw_box_corners(gamma_total, 0, 0)
        box_total.set_data(bx_total, by_total)
        
        # Update periodic image boxes (use reduced strain)
        for ox, oy, line in box_images:
            bx_img, by_img = draw_box_corners(gamma_reduced, ox, oy)
            line.set_data(bx_img, by_img)
        
        # Show remap indicator (flash for 3 frames)
        if remap_counter[0] < 3:
            remap_text.set_text('REMAP!')
        else:
            remap_text.set_text('')
        
        title.set_text(
            f'Time: {t:.4f}  |  '
            f'Total γ: {gamma_total:.4f}  |  '
            f'Reduced γ: {gamma_reduced:.4f}'
        )
        
        artists = [scatter_main, box_main, box_total, title, remap_text]
        artists.extend([sc for _, _, sc in scatter_images])
        artists.extend([line for _, _, line in box_images])
        return artists
    
    anim = FuncAnimation(fig, update, frames=len(trajectories), interval=100, blit=True)
    plt.close(fig)
    
    display(HTML(anim.to_jshtml()))
else:
    print("Not enough trajectory frames for animation. Increase n_steps or decrease traj_every.")

## 10. Extra: Green-Kubo Viscosity Analysis

For **equilibrium** simulations (no applied shear), you can compute the viscosity from stress fluctuations using the Green-Kubo relation:

$$\eta = \frac{V}{k_B T} \int_0^\infty \langle \sigma_{xy}(t) \, \sigma_{xy}(0) \rangle \, dt$$

This requires long equilibrium runs to get good statistics on the stress autocorrelation function.

In [ ]:
# Example: stress autocorrelation (more useful for equilibrium runs)
if len(sigma_xy) > 100:
    # Compute autocorrelation
    acf = rheo.autocorrelation_fft(sigma_xy)
    time_lags = times[:len(acf)] - times[0]
    
    # Plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(time_lags[:len(acf)//2], acf[:len(acf)//2], 'b-')
    ax.set_xlabel('Time lag')
    ax.set_ylabel('Stress ACF')
    ax.set_title('Stress Autocorrelation Function')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data points for ACF analysis.")

## 11. Saving Results

Save the stress data to a file for later analysis.

In [ ]:
# Create output directory
out_dir = 'output/rpy_shear_tutorial'
os.makedirs(out_dir, exist_ok=True)

# Save stress data
stress_file = os.path.join(out_dir, 'stress.dat')
header = 'time strain sigma_xx sigma_xy sigma_xz sigma_yx sigma_yy sigma_yz sigma_zx sigma_zy sigma_zz'
stress_flat = stresses.reshape(len(stresses), -1)
data = np.column_stack([times, strains, stress_flat])
np.savetxt(stress_file, data, header=header, fmt='%.6e')
print(f"Stress data saved to {stress_file}")

# Save simulation parameters
params_file = os.path.join(out_dir, 'params.txt')
with open(params_file, 'w') as f:
    f.write(f"N = {N}\n")
    f.write(f"phi = {phi}\n")
    f.write(f"L = {L}\n")
    f.write(f"a = {a}\n")
    f.write(f"eta = {eta}\n")
    f.write(f"kT = {kT}\n")
    f.write(f"shear_rate = {shear_rate}\n")
    f.write(f"peclet = {peclet}\n")
    f.write(f"dt = {dt}\n")
    f.write(f"n_steps = {n_steps}\n")
    f.write(f"xi = {xi}\n")
    f.write(f"tol = {tol}\n")
    f.write(f"rpy_rcut = {rpy_params.rcut}\n")
    f.write(f"rpy_P = {rpy_params.P}\n")
    f.write(f"rpy_M = {rpy_params.M}\n")
print(f"Parameters saved to {params_file}")